In [4]:
import pandas as pd
import zipfile
import os
import json

#Project Params

params = {}
with open("params_file.json") as param_file:
    params = json.load(param_file)


def concatenate_zips(folder_path):
    all_dataframes = []

    # Loop through every file in the directory
    for filename in os.listdir(folder_path):
        if filename.endswith(".zip"):
            file_path = os.path.join(folder_path, filename)
            #Crawl through directory a
            with zipfile.ZipFile(file_path, 'r') as z:
                for internal_file in z.namelist():
                    if internal_file.endswith(".csv"):
                        with z.open(internal_file) as f:
                            df = pd.read_csv(f)
                            df['source_file'] = filename
                            all_dataframes.append(df)
    
    if all_dataframes:
        master_df = pd.concat(all_dataframes, ignore_index=True)
        return master_df
    else:
        print("Empty Dataset Directory")
        return None

In [7]:
#Load main Data Table
full_dataframe = concatenate_zips(params["Dataset_Directory"] + "Flight Data")

C:\Users\cadek\AppData\Local\Temp\ipykernel_42816\1142411262.py:25: DtypeWarning: Columns (24) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f)


In [ ]:
#Load Lookup Table
#BTS Lookup Tables https://transtats.bts.gov/DL_SelectFields.aspx?gnoyr_VQ=FGK&QO_fu146_anzr=b0-gvzr
airport_lookup = pd.read_csv(params["Dataset_Directory"] + "Data Lookup Tables\\T_MASTER_CORD.csv", index_col="AIRPORT_ID")
airport_lookup = airport_lookup[airport_lookup['AIRPORT_IS_LATEST'] == 1 and airport_lookup['AIRPORT_IS_CLOSED'] == 0]
carrier_lookup = pd.read_csv(params["Dataset_Directory"] + "Data Lookup Tables\\Carriers.csv")
#Weather Stations https://www.ncei.noaa.gov/pub/data/noaa/isd-history.csv
carrier_lookup = pd.read_csv(params["Dataset_Directory"] + "Data Lookup Tables\\Weather_Stations.csv")

In [22]:
full_dataframe.head()

,YEAR,FL_DATE,OP_UNIQUE_CARRIER,OP_CARRIER_AIRLINE_ID,OP_CARRIER,TAIL_NUM,OP_CARRIER_FL_NUM,ORIGIN_AIRPORT_ID,ORIGIN_AIRPORT_SEQ_ID,ORIGIN_CITY_MARKET_ID,...,DIVERTED,AIR_TIME,FLIGHTS,DISTANCE,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY,source_file
0,2024,1/1/2024 12:00:00 AM,9E,20363,9E,N131EV,5225.0,10397,1039707,30397,...,0.0,33.0,1.0,164.0,NaN,NaN,NaN,NaN,NaN,T_ONTIME_MARKETING_20260320_190717.zip
1,2024,1/1/2024 12:00:00 AM,9E,20363,9E,N132EV,5115.0,11433,1143302,31295,...,0.0,89.0,1.0,651.0,NaN,NaN,NaN,NaN,NaN,T_ONTIME_MARKETING_20260320_190717.zip
2,2024,1/1/2024 12:00:00 AM,9E,20363,9E,N132EV,5414.0,11423,1142308,31423,...,0.0,87.0,1.0,533.0,NaN,NaN,NaN,NaN,NaN,T_ONTIME_MARKETING_20260320_190717.zip
3,2024,1/1/2024 12:00:00 AM,9E,20363,9E,N133EV,5284.0,12953,1295304,31703,...,0.0,103.0,1.0,722.0,NaN,NaN,NaN,NaN,NaN,T_ONTIME_MARKETING_20260320_190717.zip
4,2024,1/1/2024 12:00:00 AM,9E,20363,9E,N133EV,5341.0,10994,1099402,30994,...,0.0,83.0,1.0,641.0,NaN,NaN,NaN,NaN,NaN,T_ONTIME_MARKETING_20260320_190717.zip


In [26]:
#Find top 100 airports
dest_counts = full_dataframe['DEST_AIRPORT_ID'].value_counts().reset_index()
dest_counts.columns = ['AIRPORT_ID', 'FLIGHT_COUNT']

airport_lookup.drop_duplicates()

dest_counts['AIRPORT_ID'] = dest_counts['AIRPORT_ID'].astype(int)

dest_data = pd.merge(
    dest_counts, 
    airport_lookup, 
    left_on='AIRPORT_ID', 
    right_on='AIRPORT_ID', # Use 'Code' if it's a column, or right_index=True if it's the index
    how='left'
)

dest_data

,AIRPORT_ID,FLIGHT_COUNT,AIRPORT_SEQ_ID,AIRPORT,DISPLAY_AIRPORT_NAME,DISPLAY_AIRPORT_CITY_NAME_FULL,AIRPORT_WAC,AIRPORT_COUNTRY_NAME,AIRPORT_COUNTRY_CODE_ISO,AIRPORT_STATE_NAME,...,LATITUDE,LON_DEGREES,LON_HEMISPHERE,LON_MINUTES,LON_SECONDS,LONGITUDE,AIRPORT_START_DATE,AIRPORT_THRU_DATE,AIRPORT_IS_CLOSED,AIRPORT_IS_LATEST
0,10397,342557,1039708,ATL,Hartsfield-Jackson Atlanta International,"Atlanta, GA",34,United States,US,Georgia,...,33.636667,84.0,W,25.0,41.0,-84.428056,7/1/2025 12:00:00 AM,NaN,0,1
1,13930,318122,1393008,ORD,Chicago O'Hare International,"Chicago, IL",41,United States,US,Illinois,...,41.976944,87.0,W,54.0,29.0,-87.908056,10/1/2022 12:00:00 AM,NaN,0,1
2,11298,315680,1129806,DFW,Dallas/Fort Worth International,"Dallas/Fort Worth, TX",74,United States,US,Texas,...,32.897222,97.0,W,2.0,16.0,-97.037778,12/1/2017 12:00:00 AM,NaN,0,1
3,11292,313280,1129203,DEN,Denver International,"Denver, CO",82,United States,US,Colorado,...,39.861111,104.0,W,40.0,19.0,-104.671944,7/1/2025 12:00:00 AM,NaN,0,1
4,11057,252408,1105704,CLT,Charlotte Douglas International,"Charlotte, NC",36,United States,US,North Carolina,...,35.213056,80.0,W,57.0,4.0,-80.951111,7/1/2025 12:00:00 AM,NaN,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
354,11997,90,1199702,GST,Gustavus Airport,"Gustavus, AK",1,United States,US,Alaska,...,58.425278,135.0,W,42.0,27.0,-135.707500,7/1/2011 12:00:00 AM,NaN,0,1
355,15582,52,1558203,VEL,Vernal Regional,"Vernal, UT",87,United States,US,Utah,...,40.436111,109.0,W,30.0,41.0,-109.511389,12/1/2017 12:00:00 AM,NaN,0,1
356,13282,38,1328207,MGW,Morgantown Municipal Walter L Bill Hart Field,"Morgantown, WV",39,United States,US,West Virginia,...,39.643611,79.0,W,55.0,3.0,-79.917500,5/1/2023 12:00:00 AM,NaN,0,1
357,11471,32,1147103,EAU,Chippewa Valley Regional,"Eau Claire, WI",45,United States,US,Wisconsin,...,44.865833,91.0,W,29.0,3.0,-91.484167,7/1/2011 12:00:00 AM,NaN,0,1


In [11]:
print(type(dest_counts['AIRPORT_ID'].iloc[0]))
print(type(airport_lookup.index[0]))

<class 'numpy.int64'>
<class 'numpy.int64'>
